## ChromaDB

Store and query document embeddings using ChromaDB vector database.

> **About ChromaDB:** A lightweight vector database for storing embeddings and metadata. Enables semantic search by finding similar documents based on vector distance.

### Import Libraries

In [1]:
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path
import json

### Define Global Model Constant

In [2]:
# Global model - load once, reuse everywhere
MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

### Connect to Docker ChromaDB

In [3]:
# Connect to ChromaDB running in Docker (start with: docker-compose up -d)
client = chromadb.HttpClient(host="localhost", port="8000")

print(f"Connected to ChromaDB: {client.heartbeat()}")

Connected to ChromaDB: 1773375510555273986


### Create or Get Collection

In [4]:
# Create a collection for legal documents
collection = client.get_or_create_collection(name="legal-docs")

print(f"Collection: {collection.name}")

Collection: legal-docs


### Load and Prepare Documents

In [5]:
# Load extracted txt files from Notebook 01
from llama_index.core import SimpleDirectoryReader

TXT_DIR = Path("../data/txt")
documents = SimpleDirectoryReader(str(TXT_DIR)).load_data()

print(f"Loaded {len(documents)} documents")

Loaded 4 documents


### Prepare Data for ChromaDB

In [6]:
# Prepare documents, embeddings, and metadata for ChromaDB
docs = []
metadatas = []
ids = []

for i, doc in enumerate(documents):
    docs.append(doc.text)
    metadatas.append({
        "file_name": doc.metadata.get("file_name", "unknown"),
        "file_type": doc.metadata.get("file_type", "unknown")
    })
    ids.append(f"doc_{i}")

print(f"Prepared {len(docs)} documents for insertion")

Prepared 4 documents for insertion


### Generate Embeddings

In [7]:
# Generate embeddings for all documents
embeddings = model.encode(docs).tolist()

print(f"Generated {len(embeddings)} embeddings with {len(embeddings[0])} dimensions")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated 4 embeddings with 384 dimensions


### Add Documents to ChromaDB

In [8]:
# Clear existing data (optional - for fresh start)
client.delete_collection(collection.name)
collection = client.get_or_create_collection(name="legal-docs")

# Add documents with embeddings to collection
collection.add(
    documents=docs,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"Added {len(ids)} documents to ChromaDB")

2026-03-13 09:48:32,523 - INFO - HTTP Request: DELETE http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/legal-docs "HTTP/1.1 200 OK"
2026-03-13 09:48:32,535 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"
2026-03-13 09:48:32,539 - INFO - HTTP Request: GET http://localhost:8000/api/v2/pre-flight-checks "HTTP/1.1 200 OK"
2026-03-13 09:48:32,597 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/add "HTTP/1.1 201 Created"


Added 4 documents to ChromaDB


### Query ChromaDB

In [9]:
# Query: Find similar documents
query = "What are the forest conservation issues?"

# Generate query embedding
query_embedding = model.encode([query]).tolist()

# Query ChromaDB for top 3 similar documents
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"Query: {query}\n")
print(f"Found {len(results['documents'][0])} results")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 09:48:32,658 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/query "HTTP/1.1 200 OK"


Query: What are the forest conservation issues?

Found 3 results


### Display Query Results

In [10]:
# Print query results with distances
for i, (doc, metadata, distance) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
), 1):
    print(f"\n{i}. Distance: {distance:.4f}")
    print(f"   Source: {metadata['file_name']}")
    print(f"   Text: {doc[:200]}...")


1. Distance: 1.4342
   Source: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
   Text: A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025

A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025



Author: Vikram ...

2. Distance: 1.5246
   Source: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt
   Text: A John Kennedy vs The State Of Tamil Nadu on 24 March, 2025
Author: Vikram Nath
Bench: Vikram Nath
2025 INSC 443                                                             REPORTABLE
                ...

3. Distance: 1.7886
   Source: A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt
   Text: A Rajendra vs Gonugunta Madhusudhan Rao on 4 April, 2025

A Rajendra vs Gonugunta Madhusudhan Rao on 4 April, 2025

A Rajendra vs Gonugunta Madhusudhan Rao on 4 April, 2025



Author: Abhay S. Oka Ben...


### Try Different Queries

In [11]:
# Test multiple queries
queries = [
    "What is the Supreme Court judgment about?",
    "Tiger reserve protection laws",
    "Tea estate workers rehabilitation"
]

for query in queries:
    query_embedding = model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=1,
        include=["documents", "metadatas"]
    )
    print(f"\nQuery: {query}")
    print(f"Top result from: {results['metadatas'][0][0]['file_name']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 09:48:32,803 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/query "HTTP/1.1 200 OK"



Query: What is the Supreme Court judgment about?
Top result from: A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 09:48:32,840 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/query "HTTP/1.1 200 OK"



Query: Tiger reserve protection laws
Top result from: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-03-13 09:48:32,881 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/query "HTTP/1.1 200 OK"



Query: Tea estate workers rehabilitation
Top result from: A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt


### Collection Statistics

In [12]:
# Get collection count
count = collection.count()

print("=== ChromaDB Collection Stats ===")
print(f"Collection name: {collection.name}")
print(f"Total documents: {count}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"Model: {MODEL_NAME}")

2026-03-13 09:48:32,906 - INFO - HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/count?read_level=index_and_wal "HTTP/1.1 200 OK"


=== ChromaDB Collection Stats ===
Collection name: legal-docs
Total documents: 4
Embedding dimensions: 384
Model: all-MiniLM-L6-v2


### List All Documents in Collection

In [13]:
# Get all document metadata
all_data = collection.get(include=["metadatas"])

print("Documents in ChromaDB:")
for i, metadata in enumerate(all_data['metadatas'], 1):
    print(f"  {i}. {metadata['file_name']}")

2026-03-13 09:48:33,049 - INFO - HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/2f1d885f-f691-4d89-a332-0fb1f7d2fa06/get "HTTP/1.1 200 OK"


Documents in ChromaDB:
  1. A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_docx.txt
  2. A_John_Kennedy_vs_The_State_Of_Tamil_Nadu_on_24_March_2025_1_extracted_from_pdf.txt
  3. A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_docx.txt
  4. A_Rajendra_vs_Gonugunta_Madhusudhan_Rao_on_4_April_2025_1_extracted_from_pdf.txt
